In [21]:
import masknmf
import fastplotlib as fpl
import os
import sys
import numpy as np
import torch
from wfield import *
import wfield
from typing import *

from one.api import ONE
from brainbox.io.one import SessionLoader

%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
one = ONE()

In [16]:
one.list_datasets(eid="3158300f-e72c-42fc-ac6c-c981615fe00f")

['_ibl_experiment.description.yaml',
 'alf/#2025-07-29#/_ibl_trials.goCueTrigger_times.npy',
 'alf/#2025-07-29#/_ibl_trials.quiescencePeriod.npy',
 'alf/#2025-07-29#/_ibl_trials.stimOff_times.npy',
 'alf/#2025-07-29#/_ibl_trials.table.pqt',
 'alf/_ibl_bodyCamera.times.npy',
 'alf/_ibl_leftCamera.times.npy',
 'alf/_ibl_rightCamera.times.npy',
 'alf/_ibl_trials.goCueTrigger_times.npy',
 'alf/_ibl_trials.quiescencePeriod.npy',
 'alf/_ibl_trials.stimOff_times.npy',
 'alf/_ibl_trials.table.pqt',
 'alf/_ibl_wheel.position.npy',
 'alf/_ibl_wheel.timestamps.npy',
 'alf/_ibl_wheelMoves.intervals.npy',
 'alf/_ibl_wheelMoves.peakAmplitude.npy',
 'alf/widefield/imaging.imagingLightSource.npy',
 'alf/widefield/imaging.times.npy',
 'alf/widefield/imagingLightSource.properties.htsv',
 'alf/widefield/widefieldChannels.frameAverage.npy',
 'alf/widefield/widefieldLandmarks.dorsalCortex.json',
 'alf/widefield/widefieldSVT.haemoCorrected.npy',
 'alf/widefield/widefieldSVT.uncorrected.npy',
 'alf/widefield

In [22]:
sl = SessionLoader(eid = "3158300f-e72c-42fc-ac6c-c981615fe00f", one=one)
sl.load_trials()

sl.trials.keys()

(S3) /data/lab/IBL_Alyx/zadorlab/Subjects/CSK-im-001/2021-01-19/001/alf/#2025-07-29#/_ibl_trials.goCueTrigger_times.npy: 100%|████████████████████████████████████████| 6.98k/6.98k [00:00<00:00, 97.1kB/s]
(S3) /data/lab/IBL_Alyx/zadorlab/Subjects/CSK-im-001/2021-01-19/001/alf/#2025-07-29#/_ibl_trials.quiescencePeriod.npy: 100%|███████████████████████████████████████████| 6.98k/6.98k [00:00<00:00, 107kB/s]
(S3) /data/lab/IBL_Alyx/zadorlab/Subjects/CSK-im-001/2021-01-19/001/alf/#2025-07-29#/_ibl_trials.stimOff_times.npy: 100%|█████████████████████████████████████████████| 6.98k/6.98k [00:00<00:00, 71.9kB/s]
(S3) /data/lab/IBL_Alyx/zadorlab/Subjects/CSK-im-001/2021-01-19/001/alf/#2025-07-29#/_ibl_trials.table.pqt: 100%|██████████████████████████████████████████████████████| 66.8k/66.8k [00:00<00:00, 881kB/s]


Index(['goCueTrigger_times', 'quiescencePeriod', 'stimOff_times',
       'goCue_times', 'response_times', 'choice', 'stimOn_times',
       'contrastLeft', 'contrastRight', 'probabilityLeft', 'feedback_times',
       'feedbackType', 'rewardVolume', 'firstMovement_times', 'intervals_0',
       'intervals_1'],
      dtype='object')

In [23]:
sl.trials['stimOn_times']

0        54.999989
1        58.580486
2        61.783667
3        65.063690
4        69.799549
          ...     
851    4308.457158
852    4325.673830
853    4343.676099
854    4358.489819
855    4367.092586
Name: stimOn_times, Length: 856, dtype: float64

In [2]:

class ucla_wf_singlechannel(masknmf.ArrayLike):
    def __init__(self,
                 my_memmap: np.memmap,
                 dtype = np.uint16,
                 channel: int = 0,
                 mask: Optional[np.ndarray] = None,
                 num_frames: Optional[int] = None):
        # print(f"type of my_memmap is {type(my_memmap)}")
        self._channel = channel
        self._dtype = dtype
        self._mmap = my_memmap[:num_frames] if num_frames is not None else my_memmap
        self._shape = (self._mmap.shape[0], self._mmap.shape[2], self._mmap.shape[3])
        self._mask = mask.astype(dtype) if mask is not None else None

    @property
    def shape(self):
        return self._shape

    @property
    def dtype(self):
        return self._dtype

    @property
    def ndim(self):
        return 3

    def __getitem__(self, item: Union[int, list, np.ndarray, Tuple[Union[int, np.ndarray, slice, range]]]) -> np.ndarray:
        # return self._mmap[item].copy()
        if isinstance(item, (int, slice, np.ndarray, range)):
            return np.asarray(self._mmap[item, self._channel, :, :]).copy()
        elif isinstance(item, list) or isinstance(item, tuple):
            if len(item) == 1:
                return np.asarray(self._mmap[item[0], self._channel, :, :]).copy()
            elif len(item) == 2:
                return np.asarray(self._mmap[item[0], self._channel, item[1], :]).copy()
            elif len(item) == 3:
                return np.asarray(self._mmap[item[0], self._channel, item[1], item[2]]).copy()


In [3]:
u = np.load("/data/lab/IBL_Alyx/zadorlab/Subjects/CSK-im-001/2021-01-19/001/alf/widefield/widefieldU.images.npy")
v = np.load("/data/lab/IBL_Alyx/zadorlab/Subjects/CSK-im-001/2021-01-19/001/alf/widefield/widefieldSVT.uncorrected.npy")
hemocorr_v = np.load("/data/lab/IBL_Alyx/zadorlab/Subjects/CSK-im-001/2021-01-19/001/alf/widefield/widefieldSVT.haemoCorrected.npy")
frame_avg = np.load("/data/lab/IBL_Alyx/zadorlab/Subjects/CSK-im-001/2021-01-19/001/alf/widefield/widefieldChannels.frameAverage.npy")
video_obj =  wfield.load_stack("/data/lab/IBL_Alyx/zadorlab/Subjects/CSK-im-001/2021-01-19/001/raw_widefield_data/", nchannels=2)
mask = np.load("/data/lab/IBL_Alyx/zadorlab/Subjects/CSK-im-001/2021-01-19/001/raw_widefield_data/manual_mask.npy")

In [4]:
# u_gcamp = u * frame_avg[functional_channel, :, :][:, :, None]
# u_blood = u * frame_avg[1-functional_channel, :, :][:, :, None]

# u_gcamp = torch.from_numpy(u_gcamp.reshape((-1, u_gcamp.shape[2]))).to('cuda').float().to_sparse()
# u_blood = torch.from_numpy(u_blood.reshape((-1, u_blood.shape[2]))).to('cuda').float().to_sparse()

u_sparse = torch.from_numpy(u.reshape((-1, u.shape[2]))).to('cuda').float().to_sparse()
frame_avg = torch.from_numpy(frame_avg).to('cuda').float()
v = torch.from_numpy(v).to('cuda').float()
hemocorr_v = torch.from_numpy(hemocorr_v).to('cuda').float()

In [5]:
functional_channel = 1
v_blood = v[:, 1-functional_channel::2]
v_gcamp = v[:, functional_channel::2]
stack_shape = v_blood.shape[1], u.shape[0], u.shape[1]


joao_blood = masknmf.PMDArray(stack_shape,
                            u_sparse,
                            v_blood,
                            frame_avg[1-functional_channel, :, :],
                            frame_avg[1-functional_channel, :, :],
                            rescale = True)

joao_gcamp = masknmf.PMDArray(stack_shape,
                            u_sparse,
                            v_gcamp,
                            frame_avg[functional_channel, :, :],
                            frame_avg[functional_channel, :, :],
                            rescale = True)

joao_hemocorr = masknmf.PMDArray(stack_shape,
                                 u_sparse,
                                 hemocorr_v,
                                 frame_avg[functional_channel, :, :],
                                 frame_avg[functional_channel, :, :],
                                 rescale = True)

In [6]:
joao_hemocorr.to('cuda')
joao_gcamp.to('cuda')
joao_blood.to('cuda')

# Load the results from your pipeline

In [9]:
gcamp_amol = masknmf.PMDArray.from_hdf5("outputs_001_fc_1/gcamp.hdf5")
gcamp_amol.rescale = True

In [10]:
gcamp_amol.to('cuda')

# Now let's load the raw stacks and compare

In [7]:
gcamp_channel = ucla_wf_singlechannel(video_obj,
                                      channel=functional_channel,
                                      mask=mask,
                                      num_frames=None)[:]


In [8]:
joao_gcamp_gui = masknmf.PMDWidget(gcamp_channel, joao_gcamp, mean_subtract = True, include_diagnostics=False, device='cuda')
joao_gcamp_gui.show()

RFBOutputContext()

Max vertex attribute stride unknown. Assuming it is 2048
/data/home/app2139/wfield/wfieldvenv/lib/python3.11/site-packages/fastplotlib/graphics/features/_base.py:18: UserWarning: casting float64 array to float32
  warn(f"casting {array.dtype} array to float32")


RFBOutputContext()

Error during handling pointer_up event
Traceback (most recent call last):
  File "/data/home/app2139/wfield/wfieldvenv/lib/python3.11/site-packages/rendercanvas/_coreutils.py", line 50, in log_exception
    yield
  File "/data/home/app2139/wfield/wfieldvenv/lib/python3.11/site-packages/pygfx/objects/_events.py", line 395, in handle_event
    callback(event)
  File "/data/home/app2139/masknmf-toolbox/masknmf/visualization/interactive_guis.py", line 266, in end_resize
    self._crop_and_display()
  File "/data/home/app2139/masknmf-toolbox/masknmf/visualization/interactive_guis.py", line 271, in _crop_and_display
    pmd_temporal = self.pmd_stack[:, self._row_slice, self._col_slice].mean(axis=(1, 2))
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/data/home/app2139/wfield/wfieldvenv/lib/python3.11/site-packages/numpy/_core/_methods.py", line 123, in _mean
    rcount = _count_reduce_items(arr, axis, keepdims=keepdims, where=where)
         

In [12]:
amol_gcamp_gui = masknmf.PMDWidget(gcamp_channel, gcamp_amol, mean_subtract = True, include_diagnostics=False, device='cuda')
joao_gcamp_gui.show()

In [ ]:
blood_channel = ucla_wf_singlechannel(video_obj,
                                      channel=1 - functional_channel,
                                      mask=mask,
                                      num_frames=None)